# 📊 月營收 YoY 回測分析｜教材版

> **核心問題：當某股票月營收年增率（YoY）超過門檻時，買入持有 N 天的勝率有多少？**

## 如何把這份教材存成自己的版本？

**方法一：存到 Google 雲端（Colab 用戶）**
1. 點上方選單「**檔案**」
2. 選「**在雲端硬碟中儲存副本**」
3. 開啟雲端硬碟中的副本，即可自由修改與執行

**方法二：下載到本機用 Jupyter Notebook 開啟**
1. 點上方選單「**檔案**」→「**下載**」→「**下載 .ipynb**」
2. 在 Jupyter Notebook 開啟該檔案即可

---

## 資料來源

- 月營收：[FinMind](https://finmindtrade.com/)（台灣開源金融資料平台）
- 股價：[yfinance](https://pypi.org/project/yfinance/)

## 完整執行流程

```
Step 1：安裝套件
Step 2：設定參數（股票代號、YoY 門檻、持有天數）
Step 3：從 FinMind 抓取月營收，計算 YoY
Step 4：計算公告日，對齊股價資料
Step 5：找出觸發點，計算勝率與平均報酬
Step 6：視覺化（K 線圖 + YoY 走勢圖）
Step 7：多時間窗口比較（5 / 10 / 20 / 30 天）
```

> ⚠️ 本教材僅供學術研究與程式教學用途，不構成任何投資建議。


## Step 1｜安裝套件

這格安裝所需的套件，已安裝過直接跑也沒問題。

In [ ]:
!pip install requests pandas yfinance matplotlib -q
print("✅ 套件安裝完成！")


## Step 2｜設定參數

這格設定所有可調整的參數，改這裡就好，後面的 Cell 不需要動。

> 💡 **FinMind Token**：不填也可以抓資料，但每天有請求次數限制（約 600 次）。
> 若需要更多次數，可到 [finmindtrade.com](https://finmindtrade.com) 免費註冊取得 Token。


In [ ]:
# ── 可調整參數 ────────────────────────────────────────
STOCK_ID       = "2330"        # 股票代號（不含 .TW）
MARKET     = "TW"         # 上市填 TW，上櫃填 TWO
YOY_THRESHOLD  = 20            # YoY 門檻（%），超過才觸發
HOLD_DAYS      = 20            # 持有交易日數
START_DATE     = "2015-01-01"  # 資料起始日
END_DATE       = "2025-12-31"  # 資料結束日
FINMIND_TOKEN  = ""            # FinMind Token（選填）
WINDOWS        = [5, 10, 20, 30]  # 多時間窗口比較
# ─────────────────────────────────────────────────────

print(f"✅ 參數設定完成！")
print(f"   股票：{STOCK_ID}，YoY 門檻：{YOY_THRESHOLD}%，持有：{HOLD_DAYS} 天")


## Step 3｜從 FinMind 抓取月營收，計算 YoY

這格從 FinMind 抓取指定股票的完整月營收歷史，並計算每個月的 YoY（年增率）。

> 💡 YoY 計算方式：本月營收 ÷ 去年同月營收 - 1，用 `pct_change(12)` 計算。


In [ ]:
import requests
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

# 抓月營收
params = {
    "dataset":    "TaiwanStockMonthRevenue",
    "data_id":    STOCK_ID,
    "start_date": START_DATE,
    "end_date":   END_DATE,
}
if FINMIND_TOKEN:
    params["token"] = FINMIND_TOKEN

resp = requests.get("https://api.finmindtrade.com/api/v4/data", params=params, timeout=30)
data = resp.json()

if data.get("status") != 200:
    print(f"❌ 抓取失敗：{data.get('msg')}")
else:
    rev = pd.DataFrame(data["data"])
    rev["date"] = pd.to_datetime(rev["date"])
    rev = rev.sort_values("date").reset_index(drop=True)

    # 計算 YoY
    # FinMind 的 date 欄位是公告日期（次月初），往前推一個月才是真正的營收月份
    rev["revenue_month"] = rev["date"] - pd.DateOffset(months=1)
    rev["revenue_yoy"] = rev["revenue"].pct_change(12) * 100
    rev = rev.dropna(subset=["revenue_yoy"]).reset_index(drop=True)

    print(f"✅ 月營收資料抓取成功！共 {len(rev)} 筆（{rev['date'].min().date()} ～ {rev['date'].max().date()}）")
    print(f"\n最新 5 筆：")
    print(rev[["date", "revenue", "revenue_yoy"]].tail().to_string(index=False))


## Step 4｜計算公告日，對齊股價資料

台灣法規要求每月營收必須在**次月 10 日前**公告，因此：

- 1 月的營收 → 2 月 10 日才知道 → 2 月 10 日（或最近交易日）買入
- 以此類推

> ⚠️ 若 10 日剛好是假日，自動往後找最近的交易日。


In [ ]:
import yfinance as yf

# 下載股價
ticker = f"{STOCK_ID}.{MARKET}"
price_df = yf.download(ticker, start=START_DATE, end=END_DATE, auto_adjust=True, progress=False)
if isinstance(price_df.columns, pd.MultiIndex):
    price_df.columns = price_df.columns.get_level_values(0)
price_df = price_df[["Close"]].copy()

# 計算公告日（次月 10 日）
def get_announce_date(revenue_date, price_index):
    # next month 10th, find nearest trading day if holiday
    import datetime
    next_month = revenue_date + pd.DateOffset(months=1)
    announce = pd.Timestamp(next_month.year, next_month.month, 10)
    # 往後找最近交易日
    future = price_index[price_index >= announce]
    return future[0] if len(future) > 0 else None

rev["announce_date"] = rev["date"].apply(lambda d: get_announce_date(d, price_df.index))
rev = rev.dropna(subset=["announce_date"]).reset_index(drop=True)

print(f"✅ 公告日計算完成！")
print(f"   股價資料：{price_df.index.min().date()} ～ {price_df.index.max().date()}，共 {len(price_df)} 個交易日")
print(f"\n範例（最新 5 筆）：")
print(rev[["date", "revenue_yoy", "announce_date"]].tail().to_string(index=False))


## Step 5｜找出觸發點，計算勝率與平均報酬

這格篩選出 YoY 超過門檻的月份，計算在公告日買入、持有 N 天後的勝率。


In [ ]:
# 篩選觸發點
signals = rev[rev["revenue_yoy"] > YOY_THRESHOLD].copy()
print(f"YoY > {YOY_THRESHOLD}% 的月份共 {len(signals)} 次")

# 計算報酬
results = []
for _, row in signals.iterrows():
    buy_date = row["announce_date"]
    buy_idx  = price_df.index.get_loc(buy_date)
    sell_idx = buy_idx + HOLD_DAYS
    if sell_idx < len(price_df):
        buy_price  = price_df["Close"].iloc[buy_idx]
        sell_price = price_df["Close"].iloc[sell_idx]
        ret = (sell_price - buy_price) / buy_price * 100
        results.append({
            "revenue_month": row["date"],
            "announce_date": buy_date,
            "yoy": row["revenue_yoy"],
            "buy_price": buy_price,
            "sell_price": sell_price,
            "return_pct": ret
        })

result_df = pd.DataFrame(results)

if result_df.empty:
    print("❌ 無有效樣本，請調整參數")
else:
    win_rate = (result_df["return_pct"] > 0).mean() * 100
    avg_ret  = result_df["return_pct"].mean()
    print(f"\n{'='*45}")
    print(f"  股票：{STOCK_ID}　YoY 門檻：{YOY_THRESHOLD}%　持有：{HOLD_DAYS} 天")
    print(f"  樣本數：{len(result_df)}")
    print(f"  勝率：{win_rate:.1f}%")
    print(f"  平均報酬：{avg_ret:+.2f}%")
    print(f"{'='*45}")
    print(f"\n各年觸發次數：")
    print(result_df["revenue_month"].dt.year.value_counts().sort_index().to_string())


## Step 6｜視覺化

上方圖：K 線走勢，紅色▼標出每個觸發點（買入日）
下方圖：月營收 YoY 走勢，橘色虛線為門檻


In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import platform

if platform.system() == "Windows":
    matplotlib.rc("font", family="Microsoft JhengHei")
matplotlib.rcParams["axes.unicode_minus"] = False

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 9), sharex=False)

# ① K 線圖 + 觸發點
ax1.plot(price_df.index, price_df["Close"], color="#1f77b4", linewidth=1, label="Close Price")
if not result_df.empty:
    ax1.scatter(
        result_df["announce_date"],
        result_df["buy_price"],
        color="red", marker="v", s=80, zorder=5,
        label=f"Buy Signal (YoY>{YOY_THRESHOLD}%, n={len(result_df)})"
    )
ax1.set_title(f"{STOCK_ID} Price & Revenue YoY Trigger Points", fontsize=13)
ax1.set_ylabel("Price (TWD)")
ax1.legend()
ax1.grid(alpha=0.3)

# ② YoY 走勢
colors = ["green" if v > 0 else "red" for v in rev["revenue_yoy"]]
ax2.bar(rev["announce_date"], rev["revenue_yoy"], color=colors, width=20, alpha=0.7)
ax2.axhline(YOY_THRESHOLD, color="orange", linewidth=1.5, linestyle="--",
            label=f"Threshold {YOY_THRESHOLD}%")
ax2.axhline(0, color="black", linewidth=0.8)
ax2.set_title(f"{STOCK_ID} Monthly Revenue YoY (%)", fontsize=13)
ax2.set_ylabel("YoY (%)")
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()
print("✅ 圖表完成")


## Step 7｜多時間窗口比較

這格一次計算 5 / 10 / 20 / 30 天的勝率，方便比較哪個持有期最佳。


In [ ]:
print(f"{'='*50}")
print(f"  {STOCK_ID}　YoY > {YOY_THRESHOLD}%　多時間窗口勝率比較")
print(f"{'='*50}")
print(f"  {'持有天數':>8} {'勝率':>8} {'平均報酬':>10} {'樣本數':>8}")
print(f"  {'-'*38}")

for w in WINDOWS:
    w_results = []
    for _, row in signals.iterrows():
        buy_date = row["announce_date"]
        buy_idx  = price_df.index.get_loc(buy_date)
        sell_idx = buy_idx + w
        if sell_idx < len(price_df):
            bp = price_df["Close"].iloc[buy_idx]
            sp = price_df["Close"].iloc[sell_idx]
            w_results.append((sp - bp) / bp * 100)

    if w_results:
        wr  = sum(1 for r in w_results if r > 0) / len(w_results) * 100
        ar  = sum(w_results) / len(w_results)
        emoji = "📈" if wr > 50 else "📉"
        print(f"  {w:>5} 天後  {emoji} {wr:>5.1f}%  {ar:>+8.2f}%  {len(w_results):>6}")

print(f"{'='*50}")
print("\n✅ 多時間窗口分析完成！")
